In [1]:
# 1) Mount Drive (if not yet)
# -------------------------
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# ================== DenseNet121 + MobileNetV2 Hybrid ==================
import os, shutil, datetime, itertools
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing import image_dataset_from_directory
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import pandas as pd
from pathlib import Path
from google.colab import drive


In [3]:
# ---------------- Mount & Paths ----------------
drive.mount('/content/drive')

DRIVE_ROOT = "/content/drive/MyDrive/Colab Notebooks"
DATA_DIR  = f"{DRIVE_ROOT}/emotion/emotion_preprocessed_split70_30_aug4000"  # train & val
TEST_DIR  = f"{DRIVE_ROOT}/emotion/emotion_preprocessed/test"

LOCAL_TRAIN = "/content/data/train"
LOCAL_VAL   = "/content/data/val"
LOCAL_TEST  = "/content/data/test"

ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_DIR = f"{DRIVE_ROOT}/DL/Emotion_DenseNet121_MobileNetV2_Hybrid_70_30_{ts}"
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

def fresh_copy(src, dst):
    if os.path.exists(dst): shutil.rmtree(dst)
    shutil.copytree(src, dst)

fresh_copy(os.path.join(DATA_DIR, "train"), LOCAL_TRAIN)
fresh_copy(os.path.join(DATA_DIR, "val"),   LOCAL_VAL)
fresh_copy(TEST_DIR, LOCAL_TEST)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# ---------------- Data pipeline ----------------
IMG_SIZE_RAW = (48, 48)    # dataset images (grayscale)
IMG_SIZE_INP = (224, 224)  # model input
BATCH = 32
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE


def make_loader(dir_path, shuffle=True, batch=BATCH):
    ds = image_dataset_from_directory(
        dir_path,
        labels="inferred",
        label_mode="categorical",   # one-hot for Keras metrics/ROC
        color_mode="grayscale",
        image_size=IMG_SIZE_RAW,
        batch_size=batch,
        shuffle=shuffle,
        seed=SEED
    )
    class_names = ds.class_names

    def _prep(x, y):
        x = tf.image.grayscale_to_rgb(x)
        x = tf.image.resize(x, IMG_SIZE_INP)
        x = tf.cast(x, tf.float32) / 255.0  # keep [0,1]; branch-wise preprocess later
        return x, y

    return ds.map(_prep, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE), class_names

train_ds, class_names = make_loader(LOCAL_TRAIN, shuffle=True)
val_ds, _             = make_loader(LOCAL_VAL,   shuffle=False)
test_ds, _            = make_loader(LOCAL_TEST,  shuffle=False)
num_classes = len(class_names)
print("Classes:", class_names, "| Num classes:", num_classes)

# ---------------- Hybrid model: DenseNet121 + MobileNetV2 feature fusion ----------------
from tensorflow.keras.applications import DenseNet121, MobileNetV2
from tensorflow.keras.applications.densenet import preprocess_input as densenet_pp
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mnetv2_pp

# Backbones (frozen)
densenet_base = DenseNet121(include_top=False, weights="imagenet", input_shape=(224,224,3))
mobilenet_base = MobileNetV2(include_top=False, weights="imagenet", input_shape=(224,224,3))
densenet_base.trainable = False
mobilenet_base.trainable = False

inp = layers.Input(shape=(224,224,3), name="inp_rgb01")  # [0,1] RGB

# Branch-1: DenseNet121 preprocess & features
x_d = layers.Lambda(lambda z: densenet_pp(z*255.0), name="densenet_preproc")(inp)
f_d = densenet_base(x_d, training=False)
f_d = layers.GlobalAveragePooling2D(name="gap_densenet121")(f_d)

# Branch-2: MobileNetV2 preprocess & features
x_m = layers.Lambda(lambda z: mnetv2_pp(z*255.0), name="mobilenetv2_preproc")(inp)
f_m = mobilenet_base(x_m, training=False)
f_m = layers.GlobalAveragePooling2D(name="gap_mobilenetv2")(f_m)

# Fusion + classifier head
f = layers.Concatenate(name="concat_feats")([f_d, f_m])
f = layers.BatchNormalization()(f)
f = layers.Dense(512, activation="relu")(f)
f = layers.BatchNormalization()(f)
f = layers.Dropout(0.5)(f)
out = layers.Dense(num_classes, activation="softmax", name="pred")(f)

model = models.Model(inp, out, name="Hybrid_DenseNet121_MobileNetV2")
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

Found 28000 files belonging to 7 classes.
Found 8616 files belonging to 7 classes.
Found 7178 files belonging to 7 classes.
Classes: ['angry', 'disgusted', 'fearful', 'happy', 'neutral', 'sad', 'surprised'] | Num classes: 7
29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "Hybrid_DenseNet121_MobileNetV2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ inp_rgb01           │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ densenet_preproc    │ (None, 224, 224,  │          0 │ inp_rgb01[0][0]   │
│ (Lambda)            │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mobilenetv2_preproc │ (None, 224, 224,  │          0 │ inp_rgb01[0][0]   │
│ (Lambda)            │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ densenet121         │ (None, 7, 7,      │  7,037,504 │ densenet_preproc… │
│ (Functional)        │ 1024)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mobilenetv2_1.00_2… │ (None, 7, 7,      │  2,257,984 │ mobilenetv2_prep… │
│ (Functional)        │ 1280)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gap_densenet121     │ (None, 1024)      │          0 │ densenet121[0][0] │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gap_mobilenetv2     │ (None, 1280)      │          0 │ mobilenetv2_1.00… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat_feats        │ (None, 2304)      │          0 │ gap_densenet121[… │
│ (Concatenate)       │                   │            │ gap_mobilenetv2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 2304)      │      9,216 │ concat_feats[0][… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 512)       │  1,180,160 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 512)       │      2,048 │ dense[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 512)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pred (Dense)        │ (None, 7)         │      3,591 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 10,490,503 (40.02 MB)

 Trainable params: 1,189,383 (4.54 MB)

 Non-trainable params: 9,301,120 (35.48 MB)

In [5]:
# ---------------- Callbacks ----------------
ckpt_path = f"{OUT_DIR}/best_model.keras"  # use native Keras format
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(ckpt_path, monitor="val_accuracy", save_best_only=True, mode="max", verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
]

# ---------------- Train ----------------
EPOCHS = 15  # start modest; increase if stable
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)


Epoch 1/15
875/875 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step - accuracy: 0.3417 - loss: 2.2636
Epoch 1: val_accuracy improved from -inf to 0.50046, saving model to /content/drive/MyDrive/Colab Notebooks/DL/Emotion_DenseNet121_MobileNetV2_Hybrid_70_30_20251011_132155/best_model.keras
875/875 ━━━━━━━━━━━━━━━━━━━━ 201s 176ms/step - accuracy: 0.3418 - loss: 2.2634 - val_accuracy: 0.5005 - val_loss: 1.4561 - learning_rate: 1.0000e-04
Epoch 2/15
875/875 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step - accuracy: 0.5001 - loss: 1.5536
Epoch 2: val_accuracy improved from 0.50046 to 0.52983, saving model to /content/drive/MyDrive/Colab Notebooks/DL/Emotion_DenseNet121_MobileNetV2_Hybrid_70_30_20251011_132155/best_model.keras
875/875 ━━━━━━━━━━━━━━━━━━━━ 119s 135ms/step - accuracy: 0.5001 - loss: 1.5536 - val_accuracy: 0.5298 - val_loss: 1.3492 - learning_rate: 1.0000e-04
Epoch 3/15
875/875 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step - accuracy: 0.5484 - loss: 1.3236
Epoch 3: val_accuracy improved from 0.52983 to 0.54561, sa

In [6]:
# Save model & history
model.save(os.path.join(OUT_DIR, "hybrid_densenet121_mobilenetv2_final.keras"))
pd.DataFrame(history.history).to_csv(os.path.join(OUT_DIR, "training_history.csv"), index=False)

# ---------------- Curves ----------------
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(history.history["accuracy"], label="Train Acc")
plt.plot(history.history["val_accuracy"], label="Val Acc")
plt.title("Accuracy Curve"); plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Val Loss")
plt.title("Loss Curve"); plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "training_curves.png"), dpi=150)
plt.close()
print("Training curves saved.")

# ---------------- Evaluate on Test ----------------
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"TEST -> loss: {test_loss:.4f} | acc: {test_acc:.4f}")

# ---------------- Predictions & Reports ----------------
# true labels (one-hot)
y_true = []
for _, y in test_ds.unbatch():
    y_true.append(y.numpy())
y_true = np.array(y_true)
y_true_cls = np.argmax(y_true, axis=1)

# probs & preds
y_prob = model.predict(test_ds, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

# Classification Report
rep = classification_report(y_true_cls, y_pred, target_names=class_names, digits=4)
print(rep)
with open(os.path.join(OUT_DIR, "classification_report.txt"), "w") as f:
    f.write(rep)

# Confusion Matrix
cm = confusion_matrix(y_true_cls, y_pred)
plt.figure(figsize=(7.2,6.2))
plt.imshow(cm, interpolation="nearest", cmap="Blues")
plt.title("Confusion Matrix"); plt.colorbar()
ticks = np.arange(num_classes)
plt.xticks(ticks, class_names, rotation=45, ha="right")
plt.yticks(ticks, class_names)
th = cm.max()/2.0
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    plt.text(j, i, cm[i, j], ha="center", color="white" if cm[i, j] > th else "black", fontsize=8)
plt.ylabel("True"); plt.xlabel("Predicted")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "confusion_matrix.png"), dpi=150)
plt.close()
print("Confusion matrix saved.")

# ---------------- ROC Curves (OvR + micro) ----------------
y_true_bin = y_true  # already one-hot
num_classes = y_true_bin.shape[1]
fpr, tpr, roc_auc = {}, {}, {}
for i in range(num_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_prob[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

fpr["micro"], tpr["micro"], _ = roc_curve(y_true_bin.ravel(), y_prob.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

plt.figure(figsize=(8,6))
plt.plot(fpr["micro"], tpr["micro"], label=f"micro-average ROC (AUC = {roc_auc['micro']:.3f})", linewidth=2)
colors = ['aqua', 'darkorange', 'cornflowerblue', 'green', 'red', 'purple', 'brown']
for i, cname in enumerate(class_names):
    plt.plot(fpr[i], tpr[i], lw=2, color=colors[i % len(colors)], label=f"{cname} (AUC = {roc_auc[i]:.3f})")
plt.plot([0,1], [0,1], "k--", lw=1)
plt.xlim([0,1]); plt.ylim([0,1.05])
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("Multi-class ROC (One-vs-Rest)")
plt.legend(loc="lower right", fontsize=8)
plt.savefig(os.path.join(OUT_DIR, "roc_curves.png"), dpi=150)
plt.close()

print("ROC curves saved:", os.path.join(OUT_DIR, "roc_curves.png"))

print("✅ All outputs saved to:", OUT_DIR)

Training curves saved.
TEST -> loss: 1.2339 | acc: 0.5616
              precision    recall  f1-score   support

       angry     0.4550    0.4332    0.4439       958
   disgusted     0.5652    0.4685    0.5123       111
     fearful     0.4304    0.3262    0.3711      1024
       happy     0.7619    0.7649    0.7634      1774
     neutral     0.4864    0.5531    0.5176      1233
         sad     0.4362    0.4828    0.4583      1247
   surprised     0.7054    0.7088    0.7071       831

    accuracy                         0.5616      7178
   macro avg     0.5487    0.5339    0.5391      7178
weighted avg     0.5602    0.5616    0.5592      7178

Confusion matrix saved.
ROC curves saved: /content/drive/MyDrive/Colab Notebooks/DL/Emotion_DenseNet121_MobileNetV2_Hybrid_70_30_20251011_132155/roc_curves.png
✅ All outputs saved to: /content/drive/MyDrive/Colab Notebooks/DL/Emotion_DenseNet121_MobileNetV2_Hybrid_70_30_20251011_132155
